In [3]:
import sys
!{sys.executable} -m pip install mss

In [9]:
import sys
!{sys.executable} -m pip install pyautogui

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): s

  DEPRECATION: Building 'pygetwindow' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pygetwindow'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'pytweening' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pytweening'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'mouseinfo' using the legacy setup.py bdist_wh

In [ ]:
from dataclasses import dataclass
from typing import Tuple, List

@dataclass
class Ball:
    color: int                    # لون الكرة (ID أو enum)
    position: Tuple[float, float] # (x, y) على الشاشة

@dataclass
class BallChain:
    color: int           # لون المجموعة
    count: int           # عدد الكرات في هذه المجموعة
    balls: List[Ball]    # الكرات الفعلية (لمن يحتاج position)

@dataclass
class GameState:
    start_position: Tuple[float, float]   # بداية المسار
    end_position: Tuple[float, float]     # نهاية المسار

    frog_position: Tuple[float, float]    # مركز الضفدع
    current_ball_color: int               # لون الكرة الحالية

    chains: List[BallChain]               # السلسلة مرتبة من البداية للنهاية


In [5]:
def can_shoot(state: GameState, chain_index: int) -> bool:
    """
    يتحقق إن كان بالإمكان التصويب على هذه المجموعة بدون انسداد
    """
    frog_x, frog_y = state.frog_position

    target_ball = state.chains[chain_index].balls[0]
    tx, ty = target_ball.position

    for i, chain in enumerate(state.chains):
        if i == chain_index:
            continue

        for ball in chain.balls:
            bx, by = ball.position

            if ray_intersects_ball(
                frog_x, frog_y,
                tx, ty,
                bx, by
            ):
                return False

    return True
def compute_early_removal_bonus(state: GameState, index: int) -> float:
    balls_before = sum(c.count for c in state.chains[:index])
    return balls_before * 1.0

def compute_color_match_bonus(ball_color, group_color) -> float:
    return 20.0 if ball_color == group_color else -10.0
def evaluate_shot(state: GameState, chain_index: int) -> float:
    # 1. فحص إمكانية التصويب
    if not can_shoot(state, chain_index):
        return float('-inf')

    score = 0.0

    # 2. حذف متسلسل
    removed = compute_chain_removal(
        state.chains,
        chain_index,
        state.current_ball_color
    )
    score += removed * 50.0

    # 3. تطابق اللون
    score += compute_color_match_bonus(
        state.current_ball_color,
        state.chains[chain_index].color
    )

    # 4. حذف مبكر
    score += compute_early_removal_bonus(state, chain_index)

    # 5. الخطر
    score -= compute_danger_penalty(state, chain_index)

    return score


In [6]:
import cv2
import numpy as np
import mss
import time

def is_background_dark(image):
    """
    تفحص هذه الدالة زوايا الصورة لتعرف هل الخلفية داكنة (Full Screen) أم فاتحة (Windowed)
    """
    h, w, _ = image.shape
    # نأخذ عينات من الزوايا الأربع (10x10 بكسل)
    corners = [
        image[0:10, 0:10],          # أعلى يسار
        image[0:10, w-10:w],        # أعلى يمين
        image[h-10:h, 0:10],        # أسفل يسار
        image[h-10:h, w-10:w]       # أسفل يمين
    ]
    # حساب متوسط السطوع في الزوايا
    avg_brightness = 0
    for corner in corners:
        gray_corner = cv2.cvtColor(corner, cv2.COLOR_BGR2GRAY)
        avg_brightness += np.mean(gray_corner)
    
    avg_brightness /= 4
    print(f"DEBUG: متوسط سطوع الخلفية = {avg_brightness:.2f}")
    # إذا كان السطوع أقل من 50، فالخلفية داكنة (أسود)
    return avg_brightness < 50
def refine_game_area(screen_img, x, y, w, h):
    roi = screen_img[y:y+h, x:x+w]
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    # التحسين يعمل فقط للخلفيات الفاتحة
    _, binary = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return x, y, w, h
    largest_cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest_cnt) > (w * h * 0.5):
        rx, ry, rw, rh = cv2.boundingRect(largest_cnt)
        return x + rx, y + ry, rw, rh
    return x, y, w, h

def detect_game_final():
    with mss.mss() as sct:
        monitor = sct.monitors[1]
        screen = np.array(sct.grab(monitor))
        screen = screen[:, :, :3] 
        # 1. هل نحن في وضع Full Screen (خلفية سوداء)؟
        dark_mode = is_background_dark(screen)
        # 2. كشف التشبع (Saturation)
        hsv = cv2.cvtColor(screen, cv2.COLOR_BGR2HSV)
        s_channel = hsv[:, :, 1]
        # العتبة
        _, thresh = cv2.threshold(s_channel, 30, 255, cv2.THRESH_BINARY)
        # تنظيف
        kernel = np.ones((3, 3), np.uint8) 
        mask = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = sorted(contours, key=cv2.contourArea, reverse=True)
        final_rect = None
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 20000: continue
            x, y, w, h = cv2.boundingRect(cnt)
            ratio = w / float(h)
            # وسعنا النطاق قليلاً ليقبل الشاشات العريضة إذا لزم الأمر
            if 1.0 < ratio < 2.0:
                print(f"Found Candidate: Ratio={ratio:.2f}, DarkMode={dark_mode}")
                if dark_mode:
                    # في الوضع الليلي/الكامل: نثق بفلتر التشبع كما هو
                    # لأن السواد حول اللعبة سيتم حذفه تلقائياً (تشبعه 0)
                    final_rect = (x, y, w, h)
                    print("--> وضع Full Screen: تم اعتماد الحدود الأصلية.")
                else:
                    # في الوضع العادي: نحتاج للتحسين لإزالة العنوان
                    refined_x, refined_y, refined_w, refined_h = refine_game_area(screen, x, y, w, h)
                    final_rect = (refined_x, refined_y, refined_w, refined_h)
                    print("--> وضع النافذة: تم تطبيق التحسين.")
                break
        if final_rect:
            fx, fy, fw, fh = final_rect
            # حفظ النتائج
            preview = screen.copy()
            cv2.rectangle(preview, (fx, fy), (fx + fw, fy + fh), (0, 255, 0), 2)
            cv2.imwrite("final_detection_fullscreen.jpg", preview)
            cropped = screen[fy:fy+fh, fx:fx+fw]
            cv2.imwrite("cropped_game.jpg", cropped)
            print(f"Done! Saved cropped_game.jpg. Dims: {fw}x{fh}")
            return fx, fy, fw, fh, cropped
        else:
            print("لم يتم العثور على اللعبة.")
            return None, None, None, None, None

if __name__ == "__main__":
    time.sleep(3)
    detect_game_final()


DEBUG: متوسط سطوع الخلفية = 30.00
لم يتم العثور على اللعبة.


In [10]:
def print_heuristics(state: GameState):
    print(f"--- تقييم الأهداف (اللون الحالي: {state.current_ball_color}) ---")
    
    for i, chain in enumerate(state.chains):
        # شرطك: إذا كان اللون مختلفاً تماماً، قد لا نهتم بالتصويب المباشر حالياً
        # لكن الهيوريستيك العام يحسب النقاط لأي ضربة
        if chain.color != state.current_ball_color:
            print(f"سلسلة {i} (لون {chain.color}): لون مختلف، تقييم منخفض.")
            # يمكنك تخطيها بـ continue إذا أردت تجاهلها تماماً
        
        # حساب التقييم باستخدام دالتك
        score = evaluate_shot(state, i)
        
        status = "متاح" if score != float('-inf') else "محجوب (Blocked)"
        print(f"سلسلة {i} (لون {chain.color}): السكور = {score:.2f} | الحالة: {status}")

# لتجربة الكود
# print_heuristics(current_state)

In [11]:
# تمثيل تقريبي بناءً على الصورة المرفقة
frog_center = (400, 300) # مركز الضفدع التقريبي
current_color = 1 # لنفترض أن الكرة في فم الضفدع زرقاء (ID: 1)

# بناء السلاسل (Chains) من اليمين إلى اليسار كما تظهر في الصورة
fake_chains = [
    BallChain(color=2, count=3, balls=[Ball(2, (750, 400)), Ball(2, (730, 420)), Ball(2, (710, 440))]), # بنفسجي
    BallChain(color=1, count=2, balls=[Ball(1, (680, 480)), Ball(1, (660, 500))]), # أزرق (هدف محتمل)
    BallChain(color=3, count=4, balls=[Ball(3, (600, 550)), Ball(3, (550, 550)), Ball(3, (500, 550)), Ball(3, (450, 550))]), # أخضر
    # ... تكملة السلسلة وصولاً للنهاية
]

current_state = GameState(
    start_position=(780, 100),
    end_position=(410, 220), # الجمجمة في المركز تقريباً
    frog_position=frog_center,
    current_ball_color=current_color,
    chains=fake_chains
)

In [ ]:
import pyautogui
import math
import time

# إعدادات الأمان (Fail-safe)
pyautogui.FAILSAFE = True 

def perform_shot_precise(target_pos: Tuple[float, float], state: GameState, game_offset: Tuple[int, int] = (0, 0)):
    """
    تنفيذ الحركة كما هو مطلوب في المشروع:
    1. التحريك باتجاه الهدف لتدوير الضفدع.
    2. النقر فوق مركز الضفدع للإطلاق.
    game_offset: هي (x, y) لزاوية نافذة اللعبة على الشاشة الحقيقية
    """
    # 1. تحويل الإحداثيات من إحداثيات الصورة إلى إحداثيات الشاشة الحقيقية
    # الإزاحة ضرورية لأن اللعبة غالباً لا تبدأ من النقطة (0,0) في الشاشة
    real_target_x = target_pos[0] + game_offset[0]
    real_target_y = target_pos[1] + game_offset[1]
    
    real_frog_x = state.frog_position[0] + game_offset[0]
    real_frog_y = state.frog_position[1] + game_offset[1]

    # 2. تحريك الماوس للهدف (بدون نقر) ليدور الضفدع [دقة التنفيذ]
    # جعلنا الـ duration صفر لضمان الاستجابة السريعة المطلوبة 
    pyautogui.moveTo(real_target_x, real_target_y, duration=0)
    
    # 3. النقر على مركز الضفدع لتنفيذ الإطلاق [المتطلب رقم 4] 
    pyautogui.click(real_frog_x, real_frog_y)

def get_best_and_shoot_precise(state: GameState, game_offset: Tuple[int, int]):
    best_score = float('-inf')
    best_target = None
    
    # البحث عن أفضل هدف بناءً على الهيوريستيك الخاص بك [cite: 15, 17]
    for i in range(len(state.chains)):
        score = evaluate_shot(state, i)
        if score > best_score:
            best_score = score
            best_target = state.chains[i].balls[0].position
            
    if best_target and best_score > float('-inf'):
        print(f"إطلاق نار استراتيجي نحو: {best_target} بسكور: {best_score}")
        perform_shot_precise(best_target, state, game_offset)
    else:
        print("لا يوجد هدف متاح (ربما الطرق مسدودة أو الألوان غير متطابقة).")

In [ ]:
import pyautogui
import math

def perform_shot(target_pos: Tuple[float, float]):
    """
    target_pos: (x, y) للكرة المستهدفة
    """
    # 1. تحريك الماوس لموقع الكرة ليدور الضفدع نحوها
    # ملاحظة: يجب مراعاة إحداثيات الشاشة الحقيقية مقابل إحداثيات نافذة اللعبة
    pyautogui.moveTo(target_pos[0], target_pos[1], duration=0.1)
    
    # 2. النقر (إطلاق الكرة)
    # المشروع يطلب النقر فوق منطقة الضفدع، ولكن برمجياً النقر في اتجاه الهدف هو الأضمن
    pyautogui.click()

def get_best_and_shoot(state: GameState):
    best_score = float('-inf')
    best_target = None
    
    for i in range(len(state.chains)):
        score = evaluate_shot(state, i)
        if score > best_score:
            best_score = score
            best_target = state.chains[i].balls[0].position
            
    if best_target:
        print(f"إطلاق النار نحو الهدف بالإحداثيات: {best_target}")
        perform_shot(best_target)
    else:
        print("لا يوجد هدف متاح حالياً.")

In [ ]:
# 1. تابع البحث عن أفضل هدف (The Thinker)
def find_best_target(state: GameState):
    """
    هذا التابع يحلل الحالة الحالية للعبة ويعيد إحداثيات أفضل كرة للتصويب عليها.
    """
    best_score = float('-inf')
    best_target_pos = None
    
    # نمر على كل سلاسل الكرات الموجودة في الحالة
    for i in range(len(state.chains)):
        # استخدام التابع الهيوريستيك الخاص بك لتقييم كل سلسلة
        score = evaluate_shot(state, i)
        
        # إذا كان التقييم الحالي أفضل من السابق، نحدث الهدف
        if score > best_score:
            best_score = score
            # نأخذ إحداثيات أول كرة في هذه السلسلة كهدف
            best_target_pos = state.chains[i].balls[0].position
            
    return best_target_pos, best_score

In [ ]:
import pyautogui
import time
import mss
import numpy as np

# --- إعدادات التحكم ---
pyautogui.PAUSE = 0
pyautogui.FAILSAFE = True

def execute_precise_shot(target_pos, frog_pos, offset):
    """
    دالة مستقلة لتنفيذ الإطلاق:
    1. تحريك للهدف.
    2. نقر على الضفدع.
    3. عودة للمركز.
    """
    # تحويل الإحداثيات من لقطة الشاشة إلى إحداثيات الشاشة الحقيقية
    real_tx = target_pos[0] + offset[0]
    real_ty = target_pos[1] + offset[1]
    real_fx = frog_pos[0] + offset[0]
    real_fy = frog_pos[1] + offset[1]

    # التنفيذ
    pyautogui.moveTo(real_tx, real_ty, duration=0.1) # التوجيه
    pyautogui.click(real_fx, real_fy)               # الإطلاق (النقر على الضفدع)
    pyautogui.moveTo(real_fx, real_fy, duration=0.05) # العودة السريعة للمركز


In [ ]:
# # test
# import pyautogui
# import time

# # إعدادات السرعة
# pyautogui.PAUSE = 0.1 # تأخير بسيط جداً لضمان ثبات الأوامر
# pyautogui.FAILSAFE = True 

# def test_directions_with_return(center_x, center_y, offset=250):
#     """
#     center_x, center_y: إحداثيات مركز الضفدع (نقطة العودة)
#     offset: المسافة للجهات الأربع
#     """
#     # تحديد الاتجاهات الأربعة
#     directions = [
#         ("UP", center_x, center_y - offset),
#         ("RIGHT", center_x + offset, center_y),
#         ("DOWN", center_x, center_y + offset),
#         ("LEFT", center_x - offset, center_y)
#     ]

#     print("بدء الاختبار: 4 اتجاهات مع العودة للمركز...")
#     print("السرعة: متوسطة (0.2 ثانية للحركة).")
#     time.sleep(3)

#     for name, target_x, target_y in directions:
#         # 1. التحرك باتجاه الهدف (بسرعة متوسطة 0.2)
#         print(f"تصويب باتجاه: {name}")
#         pyautogui.moveTo(target_x, target_y, duration=0.2)
        
#         # 2. النقر للإطلاق
#         pyautogui.click()
        
#         # 3. العودة للمركز فوراً بعد الإطلاق
#         # نعود بسرعة أعلى قليلاً للمركز (0.1) لتجهيز الطلقة التالية
#         pyautogui.moveTo(center_x, center_y, duration=0.1)
        
#         # 4. انتظار ثانية واحدة قبل الاتجاه التالي كما طلبت
#         time.sleep(1)

#     print("تم الانتهاء من الاختبار.")

# # --- تشغيل الاختبار ---
# # استبدل 500, 500 بإحداثيات الضفدع الحقيقية على شاشتك
# test_directions_with_return(500, 500)

In [ ]:
def run_bot_cycle():
    # كشف النافذة (الإزاحة والأبعاد)
    offset_x, offset_y, fw, fh, _ = detect_game_final()
    if offset_x is None: return
    
    offset = (offset_x, offset_y)
    frog_pos = (fw // 2, fh // 2)

    while True:
        # أ- الحصول على لقطة الشاشة وتحويلها لبيانات (GameState)
        # state = get_current_state_from_screen(offset, fw, fh) # تابع سيتم بناؤه بـ OpenCV
        
        # ب- البحث عن الهدف باستخدام التابع المستقل
        target_coords, score = find_best_target(state)

        # ج- إذا وجدنا هدفاً صالحاً، نقوم بالإطلاق
        if target_coords and score > float('-inf'):
            print(f"تم اختيار هدف بالسكور: {score}")
            execute_precise_shot(target_coords, frog_pos, offset)
        
        # د- الانتظار المطلوب (نصف ثانية) قبل الضربة التالية
        time.sleep(0.5)

بدء الاختبار: 4 اتجاهات مع العودة للمركز...
السرعة: متوسطة (0.2 ثانية للحركة).
تصويب باتجاه: UP
تصويب باتجاه: RIGHT
تصويب باتجاه: DOWN
تصويب باتجاه: LEFT
تم الانتهاء من الاختبار.
